# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import shutil
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
import os
print(f'Path [{os.environ["PATH"]}]')

# torch_test.py
import torch
print(f"✅ Torch CUDA available: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# tf_test.py
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
import itertools 
from timeit import default_timer as timer

# Pour les modèles et leur preprocessing
from sklearn.metrics import classification_report, confusion_matrix

# Pour la visualisation des performances
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pour construire un réseau de neurone
from tensorflow.keras import Sequential, callbacks
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import Input, Flatten, GlobalAveragePooling2D, MaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Pour la transformation sur les images
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.layers import Resizing
from tensorflow.keras.layers import RandomFlip
# do not work correctly : alternative : create own CustomRandom classes
# from tensorflow.keras.layers import RandomZoom
# from tensorflow.keras.layers import RandomRotation
# from tensorflow.keras.layers import RandomBrightness
# from tensorflow.keras.layers import RandomContrast
# from tensorflow.keras.layers import RandomTranslation 
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input, VGG16

# Utilitaire d'importation des dataset d'images
from keras.utils import image_dataset_from_directory


## 1.3 ByPass of broken layers of image randomization (when used with GPU)

In [ ]:
class CustomRandomRotation(tf.keras.layers.Layer):
    def __init__(self, factor=0.2, fill_mode="REFLECT", **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
        self.fill_mode = fill_mode.upper()

    def get_config(self):
        config = super().get_config()
        config.update({"factor": self.factor, "fill_mode": self.fill_mode})
        return config

    def call(self, images, training=True):
        if not training:
            return images

        angle_max = self.factor * 3.14159265  # radians
        batch_size = tf.shape(images)[0]

        angles = tf.random.uniform(
            shape=[batch_size],
            minval=-angle_max,
            maxval=angle_max
        )

        return self._rotate(images, angles)

    def _rotate(self, images, angles):
        transform_matrices = self._get_rotation_matrices(angles)
        image_dims = tf.shape(images)[1:3]
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=images,
            transforms=transform_matrices,
            output_shape=image_dims,
            interpolation="BILINEAR",
            fill_mode=self.fill_mode,
            fill_value=0.0
        )

    def _get_rotation_matrices(self, angles):
        cos_a = tf.math.cos(angles)
        sin_a = tf.math.sin(angles)
        zero = tf.zeros_like(cos_a)

        transforms = tf.stack([
            cos_a, -sin_a, zero,
            sin_a,  cos_a, zero,
        ], axis=1)

        return tf.pad(transforms, [[0, 0], [0, 2]], constant_values=0.0)

    def compute_output_shape(self, input_shape):
        return input_shape   
     
class CustomRandomZoom(tf.keras.layers.Layer):
    def __init__(self, zoom_range=0.2, **kwargs):
        super().__init__(**kwargs)
        self.zoom_range = zoom_range

    def call(self, inputs, training=True):
        if not training:
            return inputs
        # Compute random crop box
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        zoom = tf.random.uniform([], 1.0 - self.zoom_range, 1.0 + self.zoom_range)
        new_height = tf.cast(tf.cast(height, tf.float32) / zoom, tf.int32)
        new_width = tf.cast(tf.cast(width, tf.float32) / zoom, tf.int32)
        cropped = tf.image.resize_with_crop_or_pad(inputs, new_height, new_width)
        resized = tf.image.resize(cropped, [height, width])
        return resized

    def compute_output_shape(self, input_shape):
        return input_shape
    
class CustomRandomBrightness(tf.keras.layers.Layer):
    def __init__(self, max_delta=0.2, **kwargs):
        super().__init__(**kwargs)
        self.max_delta = max_delta

    def call(self, inputs, training=True):
        if not training:
            return inputs
        return tf.image.random_brightness(inputs, self.max_delta)

    def compute_output_shape(self, input_shape):
        return input_shape
    
class CustomRandomContrast(tf.keras.layers.Layer):
    def __init__(self, lower=0.8, upper=1.2, **kwargs):
        super().__init__(**kwargs)
        self.lower = lower
        self.upper = upper

    def call(self, inputs, training=True):
        if not training:
            return inputs
        return tf.image.random_contrast(inputs, self.lower, self.upper)

    def compute_output_shape(self, input_shape):
        return input_shape
    
class CustomRandomTranslation(tf.keras.layers.Layer):
    def __init__(self, height_factor, width_factor, **kwargs):
        super().__init__(**kwargs)
        self.height_factor = self._normalize_factor(height_factor)
        self.width_factor = self._normalize_factor(width_factor)

    def _normalize_factor(self, factor):
        if isinstance(factor, (tuple, list)):
            return factor
        return (-abs(factor), abs(factor))

    def call(self, inputs, training=True):
        if not training:
            return inputs

        inputs = tf.cast(inputs, tf.float32)
        batch_size = tf.shape(inputs)[0]
        img_height = tf.cast(tf.shape(inputs)[1], tf.float32)
        img_width = tf.cast(tf.shape(inputs)[2], tf.float32)

        translations_y = tf.random.uniform(
            [batch_size],
            minval=self.height_factor[0],
            maxval=self.height_factor[1]
        ) * img_height

        translations_x = tf.random.uniform(
            [batch_size],
            minval=self.width_factor[0],
            maxval=self.width_factor[1]
        ) * img_width

        ones = tf.ones_like(translations_x)
        zeros = tf.zeros_like(translations_x)
        c0 = tf.zeros_like(translations_x)
        c1 = tf.zeros_like(translations_x)

        # Matrices 3x3 aplaties → 8 coefficients
        transforms = tf.stack([
            ones, zeros, -translations_x,
            zeros, ones, -translations_y,
            c0, c1
        ], axis=1)

        outputs = tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs,
            transforms=transforms,
            output_shape=tf.shape(inputs)[1:3],
            interpolation='BILINEAR',
            fill_mode='REFLECT',
            fill_value=0.0
        )
        return outputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "height_factor": self.height_factor,
            "width_factor": self.width_factor
        })
        return config

    def compute_output_shape(self, input_shape):
        return input_shape

# 2. Loading and Data Enrichment

In [ ]:
def classer_images_par_age(
    repertoire_images,
    age_min=None,
    age_max=None
):
    """
    Organise les images dans des sous-dossiers en fonction de l'âge extrait du nom de fichier.
    - Peut filtrer une plage d'âges : age_min à age_max
    - Réindexe les âges sélectionnés en commençant à 0
    """

    ages_valides = set()

    # Étape 1 — Parcourir tous les fichiers et collecter les âges valides
    fichiers_eligibles = []
    for racine, _, fichiers in os.walk(repertoire_images):
        for fichier in fichiers:
            if fichier.lower().endswith(".jpg"):
                try:
                    age = int(fichier.split("_")[0])
                    if ((age_min is None or age >= age_min) and
                        (age_max is None or age <= age_max)):
                        fichiers_eligibles.append((fichier, age, racine))
                        ages_valides.add(age)
                except Exception as e:
                    print(f"⚠️ Ignoré : {fichier} (erreur : {e})")

    # Étape 2 — Réindexer les âges valides
    ages_valides = sorted(ages_valides)
    mapping_ages = {age: idx for idx, age in enumerate(ages_valides)}
    print(f"🎯 Mapping des âges : {mapping_ages}")

    # Étape 3 — Déplacer les fichiers vers les bons dossiers (réindexés)
    for fichier, age, racine in fichiers_eligibles:
        nouvelle_classe = str(mapping_ages[age])
        dest_dir = os.path.join(repertoire_images, nouvelle_classe)
        os.makedirs(dest_dir, exist_ok=True)

        chemin_source = os.path.join(racine, fichier)
        chemin_destination = os.path.join(dest_dir, fichier)

        try:
            shutil.move(chemin_source, chemin_destination)
        except Exception as e:
            print(f"❌ Erreur déplacement {fichier} : {e}")

    print("✅ Organisation terminée.")

In [ ]:
data_dir = "C:\\Users\\remyc\\Downloads\\visages\\"  
min_age = 40
max_age = 45
classer_images_par_age(data_dir, min_age, max_age)

train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="training",          # Charger les données partie entraînement
    seed=42,                    # Graine pour le découpage des données
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="validation",        # Charger les données partie validation
    seed=42,                    # même Graine pour récupérer les 20% restant 
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)

In [ ]:
# Nombre de lot dans l'ensemble d'entraînement
print("Nombre de batch dans train_ds:", train_ds.cardinality().numpy())  # type: ignore
# Nombre de lot dans l'ensemble de validation
print("Nombre de batch dans val_ds:", val_ds.cardinality().numpy())  # type: ignore

In [ ]:
# Affichage aléatoire de 6 images
fig, axs = plt.subplots(2, 3, figsize=(12,8))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for images, labels in train_ds.take(1):
    for j, i in enumerate(np.random.choice(np.arange(0, len(labels)), size=6)):
        img = images[i].numpy().astype("uint8")
        axs[j].axis('off')
        # Affichage de l'image en niveaux de gris
        axs[j].imshow(img)
        # Titre avec le label
        axs[j].set_title(f'Label: {str(labels[i].numpy()+min_age)}')
plt.show()

# 3. Deep learning

>Bonnes pratiques (computer vision)
> - couches de transformation des images (redimensionnement, normalisation et augmentation)
> - couches convolutives de détection des features cachées (filtrage / bord / flou / ...)
> - couches de pooling (réduction de dimension)
> - couche de réduction des connections (pour éviter le surapprentissage)
> - couches denses d'apprentissage

## 3.1 Modèle Tensor Flow Keras (spécialisation image)

In [ ]:
class TimingCallback(Callback):
    def __init__(self, logs={}):
        self.logs=[]
    def on_epoch_begin(self, epoch, logs={}):
        self.starttime = timer()
    def on_epoch_end(self, epoch, logs={}):
        self.logs.append(timer()-self.starttime)

#### Creation & Compilation

In [ ]:
# callback optimisant le temps de traitement
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,                 # critère à observer sur 5 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on log quand on arrête prématurément
    restore_best_weights=True,
    mode='min',                 
)
reduce_lr_on_plateau = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,                 # critère à observer sur 3 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on ne log que sur l'évènement de réduction
    factor=0.1,                 # facteur de réduction si le cycle est observé
    mode='min',
)
timing = TimingCallback()

In [ ]:
# Définitions des dimension d'entrée et de sortie optimisées
for images, labels in train_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images train :", shape_image)  # type: ignore
num_classes = len(train_ds.class_names)
print("Nombre de classes train:", num_classes)

for images, labels in val_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images val :", shape_image)  # type: ignore
num_classes = len(val_ds.class_names)
print("Nombre de classes val :", num_classes)

In [ ]:
### Instanciation du modèle par application successive des layers et en utilisant le core vgg16 pre-entrainé
nn_vgg16_core = VGG16(weights='imagenet', include_top=False)
nn_vgg16_core.trainable = False
inputs = Input(shape=shape_image, name="Input")
# NB : afin de contourner les problèmes stateless qui ralentissent enormement le calcul sur GPU on doit utiliser des classes Custom
x = CustomRandomRotation(0.1)(inputs)
x = CustomRandomTranslation(0.1, 0.1)(x)
x = CustomRandomZoom(0.1)(x)
# x = CustomRandomBrightness(0.1)(x) # non recommandé pour vgg16
# x = CustomRandomBrightness(0.1)(x) # non recommandé pour vgg16
# x = CustomRandomContrast(0.1)(x) # non recommandé pour vgg16
# x = Resizing(50,50)(x) # non recommandé pour vgg16
# x = Rescaling(1./255)(x) # non recommandé pour vgg16
x = RandomFlip("horizontal")(x)
x = nn_vgg16_core(x)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(rate=0.2)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(rate=0.2)(x)
outputs = Dense(1, activation='linear')(x)

nn_tfkeras_func = Model(inputs=inputs, outputs=outputs)

In [ ]:
nn_tfkeras_func.compile(
    loss='mse',                             # fonction de perte
    optimizer='adam',                       # algorithme d'optimisation
    metrics=['mean_absolute_error'],        # métrique d'évaluation
)

In [ ]:
# preprocessing des images pour VGG16 avant application de l'entrainement
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))
training_history = nn_tfkeras_func.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=20, 
    callbacks = [
        reduce_lr_on_plateau,
        early_stopping,
        timing
    ]
)

In [ ]:
train_mae = training_history.history['mean_absolute_error']
val_mae = training_history.history['val_mean_absolute_error']
train_loss = training_history.history['loss']
val_loss = training_history.history['val_loss']
fig, axs = plt.subplots(1, 2, figsize=(12,6))
axs[0].plot(train_loss, label='Loss (training)')
axs[0].plot(val_loss, label='Loss (validation)')
axs[0].set_title('Loss evolution per epoch')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[1].plot(train_mae, label='MAE (training)')
axs[1].plot(val_mae, label='MAE (validation)')
axs[1].set_title('MAE evolution per epoch')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()
plt.show()

In [ ]:
# Prédictions du modèle : pour chaque échantillon, une regression linéaire sur l'age relatif predit
# nous commencons avec l'échelle (0 <=> min_age) qu'il faudra restituer après avoir arrondi l'age a l'entier le plus proche
y_test_prob = nn_tfkeras_func.predict(val_ds)
y_test_pred_class = np.rint(y_test_prob).astype(int).flatten()
y_test_pred_age = y_test_pred_class + min_age

y_test = []
for _, labels in val_ds.unbatch():
    y_test.append(labels.numpy())
y_test = np.array(y_test, dtype=int)
y_test_age = y_test + min_age

In [ ]:
# Étape 1 — Liste des erreurs grossières
error_indexes = []
for i in range(len(y_test_pred_age)):
    if (np.abs(y_test_pred_age[i] - y_test_age[i])>3):
        error_indexes += [i]
print(f"Nombre d'erreur grossières du modèle [{len(error_indexes)}]")

# Étape 2 — Récupération des images associées dans l'ensemble de validation
val_ds_reloaded = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="validation",        # Charger les données partie validation
    seed=42,                    # même Graine pour récupérer les 20% restant 
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)
images_list = []
for batch_images, _ in val_ds_reloaded.unbatch().batch(1).take(len(y_test_pred_age)):
    images_list.append(batch_images[0].numpy())  # batch_images est de forme (1, H, W, C)
images_array = np.array(images_list)  # (N, H, W, C)

# Étape 3 — Affichage aléatoire des images sur lesquelles le modèle s'est trompé
(max_line, max_col) = (3, 3)
print(f"On en affiche aléatoirement {max_line}x{max_col}")
fig, axs = plt.subplots(max_line, max_col, figsize=(max_line*4,max_col*4))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(error_indexes, size=max_line*max_col, replace=False)):
    axs[j].axis('off')
    # Affichage de l'image
    axs[j].imshow(images_array[i].astype("uint8"))
    # Titre avec le label
    axs[j].set_title(
        f'True Label: {str(y_test_age[i])} | '
        f'Prediction: {str(y_test_pred_age[i])}'
    )

In [ ]:
# verification avec une image perso
# === PARAMÈTRES ===
img_path = "C:\\Users\\remyc\\Downloads\\visage_perso\\25_remy.jpg"
img_size = (224, 224)     # même taille que pendant l'entraînement
model = nn_tfkeras_func   # ton modèle déjà chargé

# === PRÉTRAITEMENT ===
img = image.load_img(img_path, target_size=img_size)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # format batch (1, H, W, C)

# === PRÉDICTION ===
y_pred_scaled = model.predict(img_array)             # valeur flottante entre 0 et (max_age - min_age)
y_pred_class = np.rint(y_pred_scaled).astype(int)    # arrondi
y_pred_age = y_pred_class[0][0] + min_age

print(f"Âge prédit : {y_pred_age} ans")